# Project 03: Advanced SQL for Data Engineering

This Jupyter Notebook demonstrates production-grade advanced SQL patterns used in modern data engineering pipelines and enterprise data warehouses.

## Covered Patterns & Modules
1. **Recursive CTEs** - Hierarchical Org Chart Traversal & Subtree Rollups
2. **Window Frame Specifications** - Moving Averages (`ROWS BETWEEN`) & Running Totals
3. **Gaps and Islands** - Continuous Streak Grouping & Missing Sequence Identification
4. **Time Series Gap Filling** - Date Spines, Zero Filling & LOCF (Last Observation Carried Forward)
5. **Upsert & MERGE Patterns** - Idempotent Data Ingestion (`ON CONFLICT DO UPDATE`)
6. **CDC Stream Processing** - Deduplication & Latest Snapshot Extraction
7. **Production SQL ETL Pipeline** - Staging, Data Quality Validation, Upsert & Execution Audit Logging


In [ ]:
import sys
import os
import pandas as pd

# Add parent project path to sys.path
sys.path.append(os.path.abspath(".."))
from src.query_executor import QueryExecutor

# Initialize in-memory database
executor = QueryExecutor(db_path=":memory:")
executor.init_schema("../schemas/schema_ddl.sql")
executor.load_seed_data("../datasets/seed_data.sql")

print("QueryExecutor initialized successfully with schema and seed data.")


## 1. Recursive CTE: Organizational Hierarchy Traversal

Recursive CTEs allow querying graph or tree-structured datasets such as employee org charts, category trees, and bill-of-materials.


In [ ]:
org_sql = """
WITH RECURSIVE org_chart AS (
    SELECT
        employee_id,
        full_name,
        job_title,
        department,
        manager_id,
        0 AS depth,
        full_name AS hierarchy_path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT
        e.employee_id,
        e.full_name,
        e.job_title,
        e.department,
        e.manager_id,
        oc.depth + 1 AS depth,
        oc.hierarchy_path || ' > ' || e.full_name AS hierarchy_path
    FROM employees e
    JOIN org_chart oc ON e.manager_id = oc.employee_id
)
SELECT depth, full_name, job_title, department, hierarchy_path
FROM org_chart
ORDER BY depth, employee_id;
"""

df_org = pd.DataFrame(executor.execute_query(org_sql))
df_org


## 2. Window Frame Specs & Moving Averages

Window frame specifications (`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`) allow computing rolling moving averages and running sums over continuous partitions.


In [ ]:
window_sql = """
SELECT
    metric_date,
    store_id,
    daily_revenue,
    ROUND(AVG(daily_revenue) OVER (
        PARTITION BY store_id
        ORDER BY metric_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3d,
    SUM(daily_revenue) OVER (
        PARTITION BY store_id
        ORDER BY metric_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue
FROM daily_metrics
WHERE store_id = 101
ORDER BY metric_date;
"""

df_window = pd.DataFrame(executor.execute_query(window_sql))
df_window


## 3. Production SQL ETL Pipeline with Audit Logging

Demonstrates an enterprise-grade SQL pipeline that logs job execution telemetry, validates raw staging records, deduplicates incoming CDC stream events using `ROW_NUMBER()`, and idempotently upserts clean data into production target tables.


In [ ]:
etl_script = """
INSERT INTO etl_pipeline_audit_log (pipeline_name, start_time, status)
VALUES ('JUPYTER_ETL_DEMO', DATETIME('now'), 'RUNNING');

WITH cleaned_stg AS (
    SELECT
        txn_id, customer_id, amount, txn_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY txn_id ORDER BY txn_timestamp DESC, raw_id DESC
        ) AS rn
    FROM stg_raw_transactions
    WHERE customer_id IS NOT NULL AND amount > 0
)
INSERT INTO prod_transactions (txn_id, customer_id, amount, txn_timestamp, loaded_at)
SELECT txn_id, customer_id, amount, txn_timestamp, DATETIME('now')
FROM cleaned_stg
WHERE rn = 1
ON CONFLICT(txn_id) DO UPDATE SET
    amount = EXCLUDED.amount,
    txn_timestamp = EXCLUDED.txn_timestamp,
    loaded_at = EXCLUDED.loaded_at;

UPDATE etl_pipeline_audit_log
SET status = 'SUCCESS', end_time = DATETIME('now'), rows_processed = (SELECT COUNT(*) FROM prod_transactions)
WHERE pipeline_name = 'JUPYTER_ETL_DEMO' AND status = 'RUNNING';
"""

executor.execute_script(etl_script)

df_prod = pd.DataFrame(executor.execute_query("SELECT * FROM prod_transactions;"))
df_audit = pd.DataFrame(executor.execute_query("SELECT * FROM etl_pipeline_audit_log WHERE pipeline_name = 'JUPYTER_ETL_DEMO';"))

print("=== Target Production Table ===")
display(df_prod)

print("\n=== Audit Execution Telemetry ===")
display(df_audit)
